# InduSense — Sprint 1 : audit des données (journal de bord)

**Compétences** : C1 (jeu de données pertinent et cohérent), C2 (risques éthiques et réglementaires),
C3 (préparation et intégrité des données).

Ce notebook ne contient **pas** la logique de traitement : elle vit dans le package `src/indusense`
(testé, réutilisable, piloté par `configs/`). Ici, on exécute, on observe, on **interprète** et on **décide**.
Chaque décision est reportée dans le contrat de données correspondant (`configs/contracts/`).

| Étape | Commande équivalente | Artefacts |
|---|---|---|
| 1. Cartes d'identité | `uv run indusense identity` | `artifacts/ingestion/identities/` |
| 2. Audit + cartographie | `uv run indusense audit` | `artifacts/ingestion/audits/` |
| 3. Brouillons de contrats | `uv run indusense contract draft` | `artifacts/ingestion/contracts/drafts/` |
| 4. Contrôle des contrats | `uv run indusense validate` | `artifacts/ingestion/validations/` |

## 0. Mise en place

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from indusense.config import all_sources, load_settings
from indusense.pipeline import run_pipeline


def project_root(start: Path = Path.cwd()) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient configs/pipeline.yaml."""
    for p in [start, *start.parents]:
        if (p / "configs" / "pipeline.yaml").exists():
            return p
    raise FileNotFoundError("configs/pipeline.yaml introuvable")


ROOT = project_root()
settings = load_settings(ROOT / "configs" / "pipeline.yaml")
pd.DataFrame([{"id": s.id, "fichier": s.path.name, "format": s.resolved_format,
               "existe": s.path.exists()} for s in all_sources(settings)])

,id,fichier,format,existe
0,machine,machine.sql,sql,True
1,releves_incidents,releves_incidents.csv,csv,True
2,telemetry,telemetry.csv,csv,True


## 1. Cartes d'identité (TP1)

Un fichier par source dans `artifacts/ingestion/identities/AAAAMMJJ/<fichier>_HHMMSS.md`.

In [2]:
run = run_pipeline(settings, ["identity", "audit"])
for out in run.outputs:
    if "/identities/" in out:
        display(Markdown((ROOT / out).read_text(encoding="utf-8")))

# Carte d'identité — machine.sql

- Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\machine.sql
- Filesize : 22.41 Ko
- ProcessBeginAt: 16:56:21
- ProcessEndAt: 16:56:22
- Charset: UTF-8
- Tables Number : 2

## Table `machine`

- Rows Number : 15
- Cols Number : 11

### Column list

| Colname | Infered Type | Declared Type |
|---|---|---|
| machine_code | IDENTITY | VARCHAR(16) |
| commissioning_date | DATETIME | DATE |
| max_daily_capacity | INTEGER | INT |
| max_hourly_capacity_pieces | INTEGER | INT |
| model | CATEGORY | VARCHAR(32) |
| production_line | CATEGORY | VARCHAR(16) |
| location | CATEGORY | VARCHAR(16) |
| criticality | CATEGORY | VARCHAR(8) |
| is_active | EMPTY | BOOLEAN |
| created_at | EMPTY | TIMESTAMPTZ |
| updated_at | EMPTY | TIMESTAMPTZ |

### Numeric and date columns

| Col name | Min value | Max value | Mean value | Median value |
|---|---|---|---|---|
| commissioning_date | 2019-07-23 00:00:00 | 2025-05-25 00:00:00 | — | — |
| max_daily_capacity | 750 | 1428 | 1054.87 | 1027.00 |
| max_hourly_capacity_pieces | 47 | 89 | 65.87 | 64.00 |

## Table `maintenance`

- Rows Number : 115
- Cols Number : 11

### Column list

| Colname | Infered Type | Declared Type |
|---|---|---|
| maintenance_id | IDENTITY | INT |
| machine_code | IDENTITY | VARCHAR(16) |
| maintenance_at | DATETIME | TIMESTAMPTZ |
| maintenance_type | CATEGORY | VARCHAR(16) |
| action_type | CATEGORY | VARCHAR(32) |
| component | CATEGORY | VARCHAR(64) |
| description | TEXT | TEXT |
| related_incident_id | IDENTITY | VARCHAR(16) |
| duration_hours | FLOAT | DECIMAL(6, 2) |
| created_at | EMPTY | TIMESTAMPTZ |
| updated_at | EMPTY | TIMESTAMPTZ |

### Numeric and date columns

| Col name | Min value | Max value | Mean value | Median value |
|---|---|---|---|---|
| maintenance_at | 2025-06-15 06:00:00 UTC | 2026-05-30 22:00:00 UTC | — | — |
| duration_hours | 1.00 | 8.00 | 2.94 | 2.64 |


# Carte d'identité — releves_incidents.csv

- Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\releves_incidents.csv
- Filesize : 134.64 Ko
- ProcessBeginAt: 16:56:22
- ProcessEndAt: 16:56:22
- Charset: UTF-8
- Rows Number : 1245
- Cols Number : 18

### Column list

| Colname | Infered Type |
|---|---|
| incident_id | IDENTITY |
| date | DATETIME |
| time | TIME |
| operator_name | CATEGORY |
| machine_id | IDENTITY |
| severity | INTEGER |
| operator_badge | CATEGORY |
| comment | TEXT |
| shift | CATEGORY |
| type_surchauffe | BOOLEAN |
| type_baisse_pression | BOOLEAN |
| type_vibration | BOOLEAN |
| type_bruit_mecanique | BOOLEAN |
| type_surconsommation | BOOLEAN |
| type_blocage_mecanique | BOOLEAN |
| type_alarme_capteur | BOOLEAN |
| type_arret_urgence | BOOLEAN |
| type_defaut_qualite | BOOLEAN |

### Numeric and date columns

| Col name | Min value | Max value | Mean value | Median value |
|---|---|---|---|---|
| date | 2025-06-01 00:00:00 | 2026-06-08 00:00:00 | — | — |
| time | 00:06:00 | 23:55:00 | — | — |
| severity | 2 | 5 | 2.93 | 3.00 |


# Carte d'identité — telemetry.csv

- Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\telemetry.csv
- Filesize : 7.94 Mo
- ProcessBeginAt: 16:56:22
- ProcessEndAt: 16:56:29
- Charset: ASCII (compatible UTF-8)
- Rows Number : 135626
- Cols Number : 7

### Column list

| Colname | Infered Type |
|---|---|
| machine_id | IDENTITY |
| timestamp | DATETIME |
| temperature_c | FLOAT |
| pressure_bar | FLOAT |
| voltage_mean_v | FLOAT |
| rotation_mean_rpm | FLOAT |
| pieces_produced | INTEGER |

### Numeric and date columns

| Col name | Min value | Max value | Mean value | Median value |
|---|---|---|---|---|
| timestamp | 2025-06-01 00:00:00 | 2026-06-08 23:00:00 | — | — |
| temperature_c | 32.00 | 80.00 | 48.18 | 48.05 |
| pressure_bar | 159.98 | 215.81 | 199.77 | 199.87 |
| voltage_mean_v | 220.90 | 242.00 | 227.63 | 227.42 |
| rotation_mean_rpm | 1100.00 | 1900.00 | 1589.20 | 1590.37 |
| pieces_produced | 0 | 114 | 49.53 | 49.00 |


## 2. Audit qualité

Les constats sont classés par **dimension de qualité** (guide `docs/methodo/data_audit.md`).
🔴 bloquant · 🟠 à traiter · 🔵 information.

In [3]:
from indusense.io.readers import load_source
from indusense.profiling.profile import profile_dataset

profiles = {}
for spec in all_sources(settings):
    for ds in load_source(spec, settings).datasets:
        profiles[ds.id] = profile_dataset(ds, settings)

synthese = pd.DataFrame([{
    "jeu": p.dataset_id, "lignes": p.n_rows, "colonnes": p.n_cols,
    "doublons": p.n_duplicate_rows,
    "🔴": sum(f.severity == "error" for f in p.findings),
    "🟠": sum(f.severity == "warning" for f in p.findings),
    "🔵": sum(f.severity == "info" for f in p.findings)} for p in profiles.values()])
synthese

,jeu,lignes,colonnes,doublons,🔴,🟠,🔵
0,machine,15,11,0,3,0,1
1,maintenance,115,11,0,3,2,2
2,releves_incidents,1245,18,0,0,3,0
3,telemetry,135626,7,0,0,10,1


In [4]:
constats = pd.DataFrame([{"jeu": p.dataset_id, **vars(f)}
                         for p in profiles.values() for f in p.findings])
constats.sort_values(["severity", "jeu", "dimension"]) if not constats.empty else "Aucun constat"

,jeu,dimension,severity,column,message,rule,hypothesis,suppressed
0,machine,complétude,error,is_active,colonne entièrement vide,empty_column,,
1,machine,complétude,error,created_at,colonne entièrement vide,empty_column,,
2,machine,complétude,error,updated_at,colonne entièrement vide,empty_column,,
5,maintenance,complétude,error,related_incident_id,78.3 % de valeurs manquantes (seuil 60 %),missing_high,colonne candidate à la suppression,
9,maintenance,complétude,error,created_at,colonne entièrement vide,empty_column,,
10,maintenance,complétude,error,updated_at,colonne entièrement vide,empty_column,,
3,machine,validité,info,max_daily_capacity / max_hourly_capacity_pieces,corrélation absolue 1.000 entre les deux colonnes,redundant_columns,,
8,maintenance,exactitude,info,duration_hours,distribution asymétrique (skew=1.48) : les bor...,skewed_distribution,,
6,maintenance,unicité,info,related_incident_id,"identifiant unique sur les valeurs présentes, ...",identity_incomplete,,
17,telemetry,exactitude,info,pressure_bar,distribution asymétrique (skew=-4.16) : les bo...,skewed_distribution,,


### Interprétation des constats

Pour chaque constat 🔴/🟠, une cellule sur ce modèle. **C'est la partie évaluée** : le raisonnement,
pas le code. Exemple à adapter à tes résultats réels :

> **Constat** : `telemetry.temperature_c` — min 32.00 et max 80.00, valeurs rondes répétées.
> **Observation** : N mesures exactement à 32.00 et M à 80.00 sur 135 626 lignes.
> **Hypothèse** : plage de mesure du capteur écrêtée (valeurs réelles au-delà non enregistrées).
> **Vérification** : fiche technique du capteur / responsable OT.
> **Décision** : Bronze : conserver tel quel. Silver : ajouter un indicateur `temperature_saturee`.
> Contrat : bornes physiques `min: 0`, `max: 120` (`# TODO valider`).
> **Justification** : une saturation peut précéder une panne (signal utile) ; supprimer ces lignes
> biaiserait le modèle vers les régimes normaux.

> **Constat** :
> **Observation** :
> **Hypothèse** :
> **Vérification** :
> **Décision** :
> **Justification** :

## 3. Cartographie

Relations déclarées dans les contrats (`references`) et détectées automatiquement (colonnes de même nom).
Le diagramme Mermaid s'affiche dans GitHub et dans l'aperçu Markdown de VS Code
(extension *Markdown Preview Mermaid Support*).

In [5]:
carto = next(o for o in run.outputs if "cartographie" in o and o.endswith(".md"))
display(Markdown((ROOT / carto).read_text(encoding="utf-8")))

# Cartographie des données

PK = clé candidate (unique, sans manquant). Relations « contrat » : déclarées par `references` ; « auto » : même nom de colonne.

```mermaid
erDiagram
    MACHINE {
        IDENTITY machine_code PK
        DATETIME commissioning_date
        INTEGER max_daily_capacity PK
        INTEGER max_hourly_capacity_pieces PK
        CATEGORY model
        CATEGORY production_line
        CATEGORY location
        CATEGORY criticality
        EMPTY is_active
        EMPTY created_at
        EMPTY updated_at
    }
    MAINTENANCE {
        IDENTITY maintenance_id PK
        IDENTITY machine_code
        DATETIME maintenance_at
        CATEGORY maintenance_type
        CATEGORY action_type
        CATEGORY component
        TEXT description
        IDENTITY related_incident_id
        FLOAT duration_hours
        EMPTY created_at
        EMPTY updated_at
    }
    RELEVES_INCIDENTS {
        IDENTITY incident_id PK
        DATETIME date
        TIME time
        CATEGORY operator_name
        IDENTITY machine_id
        INTEGER severity
        CATEGORY operator_badge
        TEXT comment
        CATEGORY shift
        BOOLEAN type_surchauffe
        BOOLEAN type_baisse_pression
        BOOLEAN type_vibration
        BOOLEAN type_bruit_mecanique
        BOOLEAN type_surconsommation
        BOOLEAN type_blocage_mecanique
        BOOLEAN type_alarme_capteur
        BOOLEAN type_arret_urgence
        BOOLEAN type_defaut_qualite
    }
    TELEMETRY {
        IDENTITY machine_id
        DATETIME timestamp
        FLOAT temperature_c
        FLOAT pressure_bar
        FLOAT voltage_mean_v
        FLOAT rotation_mean_rpm
        INTEGER pieces_produced
    }
    MACHINE ||--o{ MAINTENANCE : "machine_code"
    RELEVES_INCIDENTS ||--o{ MAINTENANCE : "related_incident_id"
    MACHINE ||--o{ RELEVES_INCIDENTS : "machine_id"
    MACHINE ||--o{ TELEMETRY : "machine_id"
    RELEVES_INCIDENTS }o--o{ TELEMETRY : "machine_id"
```

| Parent | Enfant | Origine | Cardinalité | Inclusion | Orphelines |
|---|---|---|---|---|---|
| machine.machine_code | maintenance.machine_code | contrat | 1-n | 100.0% | 0 |
| releves_incidents.incident_id | maintenance.related_incident_id | contrat | 1-n | 100.0% | 0 |
| machine.machine_code | releves_incidents.machine_id | contrat | 1-n | 100.0% | 0 |
| machine.machine_code | telemetry.machine_id | contrat | 1-n | 100.0% | 0 |
| releves_incidents.machine_id | telemetry.machine_id | auto | n-n | 100.0% | 0 |


## 4. Contrats de données

Le brouillon décrit l'**observé** ; le contrat de `configs/contracts/` décrit l'**attendu**
(bornes métier, valeurs autorisées, décisions RGPD). Skill de référence : `data-contract`.

In [6]:
run_validation = run_pipeline(settings, ["draft", "validate"])
rapport = next(o for o in run_validation.outputs if "/validations/" in o and o.endswith(".md"))
display(Markdown((ROOT / rapport).read_text(encoding="utf-8")))

# Contrôle des données contre leurs contrats

_Exécuté le 27/11/2026 16:56._

| Jeu de données | Contrat | Lignes | Erreurs | Alertes | Verdict |
|---|---|---|---|---|---|
| machine | v2.2.0 (active) | 15 | 0 | 0 | ✅ conforme |
| maintenance | v0.5.0 (draft) | 115 | 0 | 0 | ✅ conforme |
| releves_incidents | v1.1.0 (active) | 1245 | 0 | 0 | ✅ conforme |
| telemetry | v0.3.0 (draft) | 135626 | 3 | 0 | ❌ non conforme |

## telemetry

|  | Règle | Colonne | Message | Nb | Exemples |
|---|---|---|---|---|---|
| 🔴 | primary_key | `machine_id,timestamp` | clé primaire non unique | 1346 | [] |
| 🔴 | required | `pressure_bar` | valeurs manquantes sur une colonne obligatoire | 995 | [] |
| 🔴 | required | `rotation_mean_rpm` | valeurs manquantes sur une colonne obligatoire | 958 | [] |


### Décisions sur les violations

| Violation | Qualification (donnée fautive / contrat faux / tolérable) | Action | Couche |
|---|---|---|---|
| | | | |

## 5. Synthèse pour la datasheet

- **Pertinence pour le cas d'usage (C1)** :
- **Risques et conformité (C2)** : données personnelles, biais de représentativité :
- **Préparation à prévoir (C3)** : règles Silver issues des décisions ci-dessus :
- **Limites de l'audit** :

Compléter `docs/datasheets/<jeu>.md` à partir de `docs/datasheets/TEMPLATE.md`.